In [ ]:
import json
import numpy as np # linear algebra
import pandas as pd
import pymongo
from pymongo import MongoClient
con = pymongo.MongoClient("mongodb://localhost:27017/")
db = con.DCSTEC
collection = db.Headlines
data = list(collection.find())
df2 = pd.DataFrame(data)

df= pd.read_csv('E:/NOUR/Documents/PFE/data/india-news-headlines.csv.zip')

#preprocess
df = df.dropna()
df = df[df['headline_category'] != 'unknown']
df=df.drop(['publish_date'],axis=1)

df = df[:100000]
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Split the data into training and testing sets
X = df['headline_text']
y = df['headline_category']
# Use train_test_split to split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create a TF-IDF vectorizer
vectorizer = TfidfVectorizer()
# Transform the training data into a TF-IDF matrix
X_train_vec = vectorizer.fit_transform(X_train)
# Create a Random Forest classifier
clf = RandomForestClassifier(verbose=1)
clf.fit(X_train_vec, y_train)
# Transform the test data into a TF-IDF matrix
X_test_vec = vectorizer.transform(X_test)
# Predict on the training set
y_train_pred = clf.predict(X_train_vec)
# Calculate the training set accuracy
train_accuracy = accuracy_score(y_train, y_train_pred)
#print(f"Training Set Accuracy: {train_accuracy:.2f}")
# Predict on the test set
y_test_pred = clf.predict(X_test_vec)
# Calculate the test set accuracy (note the corrected print statement)
test_accuracy = accuracy_score(y_test, y_test_pred)
#print(f"Test Set Accuracy: {test_accuracy:.2f}")
from sklearn.pipeline import Pipeline

# Create a pipeline
pipeline = Pipeline([
    ('vectorizer', vectorizer),
    ('clf', clf)
])
import pickle

# Save the model pipeline to a pickle file
with open('india_news_headlines_model_pipeline.pickle', 'wb') as f:
    pickle.dump(pipeline, f)
departments_keywords = {
    "Ministry of Agriculture and Farmers Welfare": [
        "agriculture", "farmers", "crops", "livestock", "dairy", "fisheries",
        "irrigation", "fertilizers", "pesticides", "seeds", "machinery",
        "markets", "prices", "credit", "insurance", "research", "extension",
        "rural development", "food security", "sustainable agriculture",
        "climate change adaptation"
    ],
    
    "Ministry of Ayush": [
        "yoga", "Ayurveda", "Unani", "Siddha", "Homeopathy", "traditional medicine",
        "complementary and alternative medicine", "wellness", "health promotion",
        "disease prevention", "treatment", "research", "education", "awareness"
    ],
    
    "Ministry of Chemicals and Fertilizers": [
        "chemicals", "fertilizers", "petrochemicals", "pharmaceuticals", "plastics",
        "rubber", "dyes", "paints", "pesticides", "explosives", "safety", "environment",
        "pollution control", "research", "development", "innovation"
    ],
    
    "Ministry of Civil Aviation": [
        "aviation", "airports", "airlines", "air traffic control", "aircraft",
        "helicopters", "air safety", "air security", "air navigation", "air transport",
        "tourism", "cargo", "trade", "investment", "employment", "growth"
    ],
    
    "Ministry of Coal": [
        "coal", "mining", "energy", "power", "coke", "coke oven gas", "tar",
        "fertilizers", "chemicals", "transportation", "safety", "environment",
        "pollution control", "research", "development"
    ],
    
    "Ministry of Commerce and Industry": [
        "trade", "industry", "exports", "imports", "foreign investment", "customs",
        "tariffs", "trade barriers", "trade negotiations", "trade agreements",
        "industrial policy", "manufacturing", "services", "MSME", "startups",
        "innovation", "employment", "growth", "development"
    ],
    
    "Ministry of Communications": [
        "telecommunications", "postal services", "broadcasting", "internet",
        "mobile phones", "broadband", "5G", "satellite communication", "radio",
        "television", "print media", "social media", "IT infrastructure", "cybersecurity",
        "e-commerce", "governance"
    ],
    
    "Ministry of Consumer Affairs, Food and Public Distribution": [
        "consumers", "food", "public distribution system", "rationing", "food safety",
        "food security", "food prices", "consumer protection", "consumer rights",
        "consumer awareness", "grievance redressal", "consumer education"
    ],
    
    "Ministry of Culture": [
        "culture", "arts", "crafts", "heritage", "languages", "literature", "music",
        "dance", "theater", "cinema", "visual arts", "museums", "archives", "festivals",
        "fairs", "tourism", "education", "awareness"
    ],
    
    "Ministry of Defence": [
        "defence", "army", "navy", "air force", "coast guard", "paramilitary forces",
        "special forces", "weapons", "ammunition", "equipment", "training", "logistics",
        "intelligence", "surveillance", "reconnaissance", "cyberwarfare", "space warfare",
        "research and development"
    ],
    
    "Ministry of Earth Sciences": [
        "earth sciences", "meteorology", "oceanography", "climate change", "weather forecasting",
        "disaster management", "remote sensing", "GIS", "hydrology", "seismology", "volcanology",
        "glaciology", "paleontology", "research and development"
    ],
    
    "Ministry of Education": [
        "education", "schools", "colleges", "universities", "teachers", "students",
        "curriculum", "pedagogy", "assessment", "research", "development", "innovation",
        "literacy", "numeracy", "skills", "employability", "higher education", "technical education"
    ],
    
    "Ministry of Electronics and Information Technology": [
        "electronics", "information technology", "IT sector", "software", "hardware",
        "semiconductors", "telecom", "digital infrastructure", "cyber security", "e-governance",
        "e-commerce", "fintech", "artificial intelligence", "machine learning", "data science",
        "blockchain", "Internet of Things", "digital economy"
    ],
    
    "Ministry of Environment, Forest and Climate Change": [
        "environment", "forest", "climate change", "pollution control", "sustainable development",
        "biodiversity conservation", "wildlife protection", "eco-tourism", "renewable energy",
        "carbon footprint", "carbon neutrality", "Paris Agreement", "climate justice"
    ],
    
    "Ministry of External Affairs": [
        "foreign affairs", "diplomacy", "international relations", "foreign policy",
        "trade relations", "investment relations", "cultural relations", "development cooperation",
        "United Nations", "G20", "BRICS", "SCO", "ASEAN", "NAM", "regional organizations",
        "bilateral and multilateral relations"
    ],
    
    "Ministry of Finance": [
        "finance", "economy", "taxation", "banking", "insurance", "securities",
        "capital markets", "public debt", "fiscal policy", "monetary policy", "foreign exchange",
        "financial inclusion", "financial literacy", "financial stability", "economic growth and development"
    ],
    
    "Ministry of Fisheries, Animal Husbandry and Dairying": [
        "fisheries", "animal husbandry", "dairying", "aquaculture", "livestock", "poultry",
        "milk", "meat", "eggs", "fish", "seafood", "animal feed", "animal health", "animal welfare",
        "research and development"
    ],
    
    "Ministry of Food Processing Industries": [
        "food processing", "food safety", "food security", "food value chain", "food technology",
        "food manufacturing", "food packaging", "food retailing", "food exports", "food imports",
        "food brands", "food labeling", "food regulation", "research and development"
    ],
    
    "Ministry of Health and Family Welfare": [
        "health", "family welfare", "hospitals", "doctors", "nurses", "paramedics",
        "public health", "infectious diseases", "non-communicable diseases", "maternal and child health",
        "nutrition", "sanitation", "hygiene", "health insurance", "health technology", "research"
    ]
} 

import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import sys
# Load the pickled model pipeline
with open('india_news_headlines_model_pipeline.pickle', 'rb') as f:
    pipeline = pickle.load(f)

# Get the sentence to predict
list_of_sentence= sys.argv[1:]
results_list = []
result3=list_of_sentence

#["Extra buses to clear tourist traffic","India; Pak exchange lists of N-plants","Mafia calls the shots in Gandhinagar too","Hoshangabad farmers have enough waterHoshangabad farmers have enough water","Meet held on cashew crop forecasting"]
#for sentence in list_of_sentence:
for index, sentence in enumerate(list_of_sentence, start=1):
    

    # Predict the top 5 headline categories with probabilities
    y_proba = pipeline.predict_proba([sentence])

    # Get the top 5 probabilities and their corresponding categories
    top_5_indices = y_proba.argsort()[0][-5:][::-1]
    top_5_categories = pipeline.classes_[top_5_indices]
    top_5_probabilities = y_proba[0][top_5_indices]

    # Print the top 5 most probable headline categories and their probabilities
    cat_list = []
    for category, probability in zip(top_5_categories, top_5_probabilities):
       # print(f"Category: {category}, Probability: {probability:.2f}")
        cat_list.append(category)

    # Define the top 5 predicted keywords
    predicted_keywords = cat_list

    # Initialize variables to keep track of the most similar department and its similarity score
    most_similar_department = None
    highest_similarity_score = -1  # Initialize with a value less than 0

    # Calculate the similarity score between predicted keywords and each department's keywords
    for department, keywords in departments_keywords.items():
        # Create TF-IDF vectors for both the predicted keywords and department keywords
        tfidf_vectorizer = TfidfVectorizer()
        tfidf_matrix = tfidf_vectorizer.fit_transform(keywords + predicted_keywords)

        # Calculate cosine similarity between the two vectors
        similarity_matrix = cosine_similarity(tfidf_matrix)

        # Get the similarity score for the predicted keywords (last 5 elements)
        similarity_score = similarity_matrix[-5:, :-5].mean()  # Assuming last 5 rows correspond to predicted keywords

        # Update the most similar department if a higher similarity score is found
        if similarity_score > highest_similarity_score:
            most_similar_department = department
            highest_similarity_score = similarity_score

    # Print the most similar department
    #print("Sentence: ",sentence)
    #print("Department: ", most_similar_department)
    result1 = sentence
    result2 = most_similar_department


    # Print the results on separate lines
    results_list.append({"result1": result1, "result2": result2, "result3": result3})
    print(json.dumps(results_list))

    #print(json.dumps({"result1": result1, "result2": result2, "result3": result3}))


from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Define the keywords for each department


# Define the top 5 predicted keywords
predicted_keywords = cat_list

# Initialize variables to keep track of the most similar department and its similarity score
most_similar_department = None
highest_similarity_score = -1  # Initialize with a value less than 0

# Calculate the similarity score between predicted keywords and each department's keywords
for department, keywords in departments_keywords.items():
    # Create TF-IDF vectors for both the predicted keywords and department keywords
    tfidf_vectorizer = TfidfVectorizer()
    tfidf_matrix = tfidf_vectorizer.fit_transform(keywords + predicted_keywords)
    
    # Calculate cosine similarity between the two vectors
    similarity_matrix = cosine_similarity(tfidf_matrix)
    
    # Get the similarity score for the predicted keywords (last 5 elements)
    similarity_score = similarity_matrix[-5:, :-5].mean()  # Assuming last 5 rows correspond to predicted keywords
    
    # Update the most similar department if a higher similarity score is found
    if similarity_score > highest_similarity_score:
        most_similar_department = department
        highest_similarity_score = similarity_score

# Print the most similar department
print("Sentence: ",sentence)
print("Department: ", most_similar_department)



[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done 100 out of 100 | elapsed: 36.0min finished
